# Glacier Monitoring & Climate Change
**PyGeoVision v2.0 — Notebook 13**

Quantify Aletsch Glacier retreat (Europe's largest) and project future ice loss under RCP climate scenarios.

```bash
pip install "pygeovision[geo,timeseries]" scipy
```

In [ ]:
import pygeovision as pgv
import numpy as np, matplotlib.pyplot as plt
from pathlib import Path; import scipy.stats as stats
import warnings; warnings.filterwarnings('ignore')

BASE_DIR = Path("./outputs/13_glacier"); BASE_DIR.mkdir(parents=True,exist_ok=True)
BBOX = (7.65, 45.92, 7.95, 46.12)
client = pgv.PyGeoVision()
print(client); print("\nStudy: Aletsch Glacier, Switzerland (24-year record 2000-2024)")

## Step 1: Multi-Decadal Imagery

In [ ]:
YEARS = [2000,2005,2010,2015,2020,2024]; scenes = {}
for yr in YEARS:
    r = client.search(bbox=BBOX,date_range=(f"{yr}-08-01",f"{yr}-09-30"),
        providers=["planetary_computer"],cloud_cover_max=10,limit=2)
    print(f"  {yr}: {len(r)} scenes")
    if r:
        dl = client.download(r[:1],output_dir=BASE_DIR/str(yr),
                              post_process=["reproject:EPSG:32632","cog"])
        if dl and dl[0].success: scenes[yr] = dl[0].path
print(f"Scenes: {len(scenes)}/{len(YEARS)}")

## Step 2: Glacier Data and Trends

In [ ]:
G = {
    "year"    :[2000,2005,2010,2015,2020,2024],
    "area_km2":[86.8,84.1,81.6,78.9,76.0,73.4],
    "vol_km3" :[25.3,24.1,22.8,21.4,20.0,18.8],
    "ela_m"   :[2940,2985,3020,3065,3110,3148],
    "mb_we"   :[0.00,-0.54,-1.12,-1.78,-2.41,-2.96],
}
years_arr = np.array(G["year"])
sl_area,int_area,r_area,_,_ = stats.linregress(years_arr, G["area_km2"])
sl_ela,int_ela,_,_,_ = stats.linregress(years_arr, G["ela_m"])

area_loss = G["area_km2"][0]-G["area_km2"][-1]
print(f"{'Year':<6} {'Area km2':>9} {'Vol km3':>9} {'ELA m':>7} {'MB m.w.e.':>11}")
print("─"*45)
for i in range(len(G["year"])):
    print(f"  {G['year'][i]:<4}  {G['area_km2'][i]:>8.1f}  {G['vol_km3'][i]:>8.1f}  {G['ela_m'][i]:>6}  {G['mb_we'][i]:>10.2f}")
print(f"\nArea lost : {area_loss:.1f} km2 ({area_loss/G['area_km2'][0]*100:.1f}%)")
print(f"Area rate : {sl_area:.2f} km2/yr  R2={r_area**2:.3f}")
print(f"ELA rise  : {G['ela_m'][-1]-G['ela_m'][0]} m  (+{sl_ela:.1f} m/yr)")

## Step 3: RCP Projections

In [ ]:
yr_proj = list(range(2025,2101))
base    = G["area_km2"][-1]
rcp45   = [max(0, base - 0.55*(yr-2024)) for yr in yr_proj]
rcp85   = [max(0, base - 0.92*(yr-2024)) for yr in yr_proj]
yr45 = next((yr_proj[i] for i,a in enumerate(rcp45) if a<5),">2100")
yr85 = next((yr_proj[i] for i,a in enumerate(rcp85) if a<5),">2100")
print(f"RCP4.5 projections (moderate emissions):")
print(f"  2050: {rcp45[yr_proj.index(2050)]:.1f} km2")
print(f"  Effectively disappeared: {yr45}")
print(f"\nRCP8.5 projections (high emissions):")
print(f"  2050: {rcp85[yr_proj.index(2050)]:.1f} km2")
print(f"  Effectively disappeared: {yr85}")

## Step 4: Visualisation

In [ ]:
fig,axes = plt.subplots(2,2,figsize=(13,9))
# Area + projections
axes[0,0].plot(G["year"],G["area_km2"],'b-o',lw=2.5,ms=8,label='Observed')
axes[0,0].plot(yr_proj,rcp45,'g-',lw=2,label='RCP4.5')
axes[0,0].plot(yr_proj,rcp85,'r-',lw=2,label='RCP8.5')
axes[0,0].fill_between(yr_proj,rcp45,rcp85,alpha=0.15,color='orange')
axes[0,0].set_ylabel("Area (km2)"); axes[0,0].set_title("Glacier Area + Projections",fontsize=11,fontweight='bold')
axes[0,0].legend(fontsize=9); axes[0,0].grid(True,alpha=0.3); axes[0,0].set_xlim(2000,2100)
# Mass balance
axes[0,1].bar(G["year"],G["mb_we"],color=['#2980B9']*6,edgecolor='white')
axes[0,1].axhline(0,color='k',lw=0.8,ls='--')
axes[0,1].set_ylabel("Cumul. MB (m w.e.)"); axes[0,1].set_title("Mass Balance Cumulative",fontsize=11,fontweight='bold')
# ELA
z = np.polyfit(years_arr,G["ela_m"],1)
yr_ext = list(range(2000,2051))
axes[1,0].plot(G["year"],G["ela_m"],'r-D',lw=2.5,ms=8,label='ELA observed')
axes[1,0].plot(yr_ext,np.poly1d(z)(yr_ext),'r--',lw=1.5,alpha=0.7,label=f'+{z[0]:.1f}m/yr')
axes[1,0].set_ylabel("ELA (m a.s.l.)"); axes[1,0].set_title("Equilibrium Line Altitude",fontsize=11,fontweight='bold')
axes[1,0].legend(fontsize=9); axes[1,0].grid(True,alpha=0.3)
# Volume
axes[1,1].plot(G["year"],G["vol_km3"],'c-s',lw=2.5,ms=8)
axes[1,1].fill_between(G["year"],15,G["vol_km3"],alpha=0.25,color='cyan')
axes[1,1].set_ylabel("Volume (km3)"); axes[1,1].set_title("Ice Volume Trend",fontsize=11,fontweight='bold')
axes[1,1].grid(True,alpha=0.3)
plt.suptitle("Aletsch Glacier — Climate Monitor 2000-2024 + Projections",fontsize=12,fontweight='bold')
plt.tight_layout(); plt.savefig(BASE_DIR/"glacier_dashboard.png",dpi=150,bbox_inches='tight'); plt.show()

## Summary

In [ ]:
print("="*60); print("NOTEBOOK 13 — Glacier Monitoring"); print("="*60)
print(f"Glacier : Aletsch, Switzerland")
print(f"Period  : 2000-2024  |  Area: 86.8->73.4 km2 (-15.5%)")
print(f"RCP4.5  : effectively gone by {yr45}")
print(f"RCP8.5  : effectively gone by {yr85}")
print("\nNext: 14_oil_spill_detection_sar.ipynb")